In [1]:
import re
import pandas as pd
import os 

notebook_path= os.getcwd()
in_dir_notifications  = os.path.abspath(os.path.join(notebook_path,"..","..","Data","notification_historical","updated_notifications.jsonl"))
in_dir_master_direction = os.path.abspath(os.path.join(notebook_path,"..","..","Data","master_directory","master_directory.jsonl"))
circulars = pd.read_json(in_dir_notifications, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines = True)

In [2]:
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")

circulars["subject_code"] = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

In [3]:
dupe_codes = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
dupe_codes[dupe_codes > 1]

subject_code
33-01-010    30
Name: id, dtype: int64

In [5]:
no_match = circulars[circulars["match_method"] == "no_match"]
print(len(no_match))
for t in no_match["title"].head(10):
    print("-", t)


KeyError: 'match_method'

In [4]:
code_matches = circulars.merge(
    master_directions.dropna(subset=["subject_code"])[["id","subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)

# where code-match succeeds but your text/title match didn't
recovered = code_matches[(code_matches["match_method"] == "no_match") & code_matches["id_master"].notna()]
print(f"{len(recovered)} previously unmatched circulars recoverable via subject_code")

# where both methods matched, do they point to the same master direction?
both = code_matches[(code_matches["match_method"] != "no_match") & code_matches["id_master"].notna()]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between text-match and code-match")

KeyError: 'match_method'